# EDA cho `RAW_SCHEMA`

Notebook này phân tích raw source pool theo `shared.schemas.RAW_SCHEMA`, đồng thời audit lệch schema với file nguồn thực tế.

Lưu ý: source file hiện có thêm cột `category_id`, nên notebook sẽ ghi nhận cột dư này nhưng chỉ dùng các cột thuộc `RAW_SCHEMA` cho thống kê chính.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'shared' / 'schemas.py').exists() and (candidate / 'dataset').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared import constants, schemas

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 1000000)

RAW_COLUMNS = list(schemas.RAW_SCHEMA.names)
TIME_COL = constants.FIELD_EVENT_TIME
PRICE_COL = 'price'
DATASET_DIR = ROOT / 'dataset'
FILES = sorted(DATASET_DIR.glob('*.csv.gz'))

ROOT, DATASET_DIR, len(FILES), [path.name for path in FILES[:3]]

## Schema audit

So sánh header thực tế của source file với `RAW_SCHEMA` để phát hiện cột dư, cột thiếu, và khác biệt thứ tự cột.

In [ ]:
def read_header(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

audit_rows = []
union_columns = set()
expected_set = set(RAW_COLUMNS)

for path in FILES:
    cols = read_header(path)
    union_columns.update(cols)
    audit_rows.append({
        'file_name': path.name,
        'source_column_count': len(cols),
        'raw_schema_columns_present': expected_set.issubset(cols),
        'extra_columns': ', '.join(sorted(set(cols) - expected_set)) or '-',
        'missing_raw_columns': ', '.join(sorted(expected_set - set(cols))) or '-',
        'source_order_matches_raw_schema': [c for c in cols if c in expected_set] == RAW_COLUMNS,
    })

schema_audit_df = pd.DataFrame(audit_rows)
extra_columns = sorted(union_columns - expected_set)
missing_columns = sorted(expected_set - union_columns)

display(schema_audit_df)
print('Extra columns in source:', extra_columns or '-')
print('Missing raw columns in source:', missing_columns or '-')
print('RAW_SCHEMA order:', RAW_COLUMNS)

## Chunked EDA

Các thống kê toàn cục được tính theo chunk để tránh load toàn bộ dữ liệu vào RAM.

Session-level metrics, duplicate check, và quantile giá được tính trên một sample ổn định từ mỗi file để notebook vẫn chạy được trên raw source lớn.

In [ ]:
CHUNK_SIZE = 250_000
SAMPLE_ROWS_PER_FILE = 100_000

def raw_chunk_reader(path):
    dtype = {col: 'string' for col in RAW_COLUMNS if col not in {TIME_COL, PRICE_COL}}
    dtype[PRICE_COL] = 'float64'

    for chunk in pd.read_csv(
        path,
        usecols=RAW_COLUMNS,
        dtype=dtype,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        chunk[TIME_COL] = pd.to_datetime(chunk[TIME_COL], utc=True, errors='coerce')
        chunk[PRICE_COL] = pd.to_numeric(chunk[PRICE_COL], errors='coerce')
        yield chunk

def collect_report(paths):
    row_counts = {}
    file_ranges = []
    missing = Counter()
    event_types = Counter()
    brands = Counter()
    categories = Counter()
    month_counts = Counter()
    day_counts = Counter()
    hour_counts = Counter()
    sample_frames = []

    price_min = np.inf
    price_max = -np.inf
    price_sum = 0.0
    price_sum_sq = 0.0
    price_count = 0
    price_null = 0
    price_non_positive = 0
    global_start = None
    global_end = None

    for path in paths:
        file_rows = 0
        file_start = None
        file_end = None
        sample_rows = 0

        for chunk in raw_chunk_reader(path):
            file_rows += len(chunk)
            missing.update(chunk.isna().sum().to_dict())
            event_types.update(chunk['event_type'].dropna().astype(str))
            brands.update(chunk['brand'].dropna().astype(str))
            categories.update(chunk['category_code'].dropna().astype(str))

            times = chunk[TIME_COL].dropna()
            if not times.empty:
                chunk_start = times.min()
                chunk_end = times.max()
                file_start = chunk_start if file_start is None else min(file_start, chunk_start)
                file_end = chunk_end if file_end is None else max(file_end, chunk_end)
                global_start = chunk_start if global_start is None else min(global_start, chunk_start)
                global_end = chunk_end if global_end is None else max(global_end, chunk_end)
                month_counts.update(times.dt.to_period('M').astype(str))
                day_counts.update(times.dt.floor('D').dt.strftime('%Y-%m-%d'))
                hour_counts.update(times.dt.hour.astype(int))

            price = chunk[PRICE_COL]
            price_null += int(price.isna().sum())
            numeric_price = price.dropna()
            if not numeric_price.empty:
                price_count += int(numeric_price.size)
                price_sum += float(numeric_price.sum())
                price_sum_sq += float((numeric_price ** 2).sum())
                price_min = min(price_min, float(numeric_price.min()))
                price_max = max(price_max, float(numeric_price.max()))
                price_non_positive += int((numeric_price <= 0).sum())

            if sample_rows < SAMPLE_ROWS_PER_FILE:
                take = chunk.head(SAMPLE_ROWS_PER_FILE - sample_rows)
                if not take.empty:
                    sample_frames.append(take.copy())
                    sample_rows += len(take)

        row_counts[path.name] = file_rows
        file_ranges.append({
            'file_name': path.name,
            'rows': file_rows,
            'start_time': file_start,
            'end_time': file_end,
        })

    sample_df = pd.concat(sample_frames, ignore_index=True) if sample_frames else pd.DataFrame(columns=RAW_COLUMNS)
    price_sample = sample_df[PRICE_COL].dropna() if not sample_df.empty else pd.Series(dtype='float64')

    return {
        'row_counts': row_counts,
        'file_ranges': pd.DataFrame(file_ranges).sort_values('file_name'),
        'missing': missing,
        'event_types': event_types,
        'brands': brands,
        'categories': categories,
        'month_counts': month_counts,
        'day_counts': day_counts,
        'hour_counts': hour_counts,
        'sample_df': sample_df,
        'price_count': price_count,
        'price_sum': price_sum,
        'price_sum_sq': price_sum_sq,
        'price_min': price_min if np.isfinite(price_min) else np.nan,
        'price_max': price_max if np.isfinite(price_max) else np.nan,
        'price_null': price_null,
        'price_non_positive': price_non_positive,
        'global_start': global_start,
        'global_end': global_end,
        'price_sample': price_sample,
    }

report = collect_report(FILES)
report['global_start'], report['global_end'], len(report['sample_df'])

In [ ]:
total_rows = sum(report['row_counts'].values())
sample_df = report['sample_df']
sample_duplicates = int(sample_df.duplicated().sum()) if not sample_df.empty else 0
price_sample = report['price_sample']
price_mean = report['price_sum'] / report['price_count'] if report['price_count'] else np.nan
price_var = (report['price_sum_sq'] / report['price_count']) - (price_mean ** 2) if report['price_count'] else np.nan
price_std = float(np.sqrt(max(price_var, 0.0))) if pd.notna(price_var) else np.nan
price_quantiles = price_sample.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).rename('price').to_frame() if not price_sample.empty else pd.DataFrame(columns=['price'])
price_p99 = float(price_quantiles.loc[0.99, 'price']) if not price_quantiles.empty else np.nan
sample_outliers = int((price_sample > price_p99).sum()) if not price_sample.empty and pd.notna(price_p99) else 0

summary_df = pd.DataFrame([
    {'metric': 'source_files', 'value': len(FILES)},
    {'metric': 'raw_columns', 'value': len(RAW_COLUMNS)},
    {'metric': 'total_rows', 'value': total_rows},
    {'metric': 'sample_rows', 'value': len(sample_df)},
    {'metric': 'global_event_time_start', 'value': report['global_start']},
    {'metric': 'global_event_time_end', 'value': report['global_end']},
    {'metric': 'price_null_rows', 'value': report['price_null']},
    {'metric': 'price_non_positive_rows', 'value': report['price_non_positive']},
])

missing_df = pd.DataFrame([
    {'field': field, 'missing_count': int(report['missing'].get(field, 0))}
    for field in RAW_COLUMNS
])
missing_df['missing_pct'] = missing_df['missing_count'] / total_rows * 100 if total_rows else np.nan
missing_df = missing_df.sort_values(['missing_count', 'field'], ascending=[False, True])

event_type_df = pd.DataFrame(report['event_types'].most_common(), columns=['event_type', 'count'])
brand_df = pd.DataFrame(report['brands'].most_common(20), columns=['brand', 'count'])
category_df = pd.DataFrame(report['categories'].most_common(20), columns=['category_code', 'count'])
month_df = pd.DataFrame(report['month_counts'].most_common(), columns=['month', 'count'])
day_df = pd.DataFrame(report['day_counts'].most_common(15), columns=['day', 'count'])
hour_df = pd.DataFrame(sorted(report['hour_counts'].items()), columns=['hour', 'count'])

price_stats_df = pd.DataFrame([
    {'stat': 'count', 'value': report['price_count']},
    {'stat': 'mean', 'value': price_mean},
    {'stat': 'std', 'value': price_std},
    {'stat': 'min', 'value': report['price_min']},
    {'stat': 'max', 'value': report['price_max']},
    {'stat': 'sample_duplicate_rows', 'value': sample_duplicates},
    {'stat': 'sample_outlier_rows_above_p99', 'value': sample_outliers},
])

session_df = pd.DataFrame()
if not sample_df.empty and 'user_session' in sample_df.columns:
    session_df = sample_df.groupby('user_session', dropna=True).agg(
        event_count=('user_session', 'size'),
        first_event_time=(TIME_COL, 'min'),
        last_event_time=(TIME_COL, 'max'),
    ).reset_index()
    session_df['duration_minutes'] = (session_df['last_event_time'] - session_df['first_event_time']).dt.total_seconds() / 60
    session_df = session_df.sort_values(['event_count', 'duration_minutes'], ascending=[False, False])

session_summary_df = pd.DataFrame([
    {'metric': 'unique_sessions_in_sample', 'value': session_df['user_session'].nunique() if not session_df.empty else 0},
    {'metric': 'unique_users_in_sample', 'value': sample_df['user_id'].nunique(dropna=True) if not sample_df.empty else 0},
    {'metric': 'unique_products_in_sample', 'value': sample_df['product_id'].nunique(dropna=True) if not sample_df.empty else 0},
    {'metric': 'median_session_events_in_sample', 'value': session_df['event_count'].median() if not session_df.empty else np.nan},
    {'metric': 'median_session_duration_minutes_in_sample', 'value': session_df['duration_minutes'].median() if not session_df.empty else np.nan},
])

display(summary_df)
display(report['file_ranges'])
display(missing_df)
display(event_type_df)
display(brand_df)
display(category_df)
display(month_df)
display(day_df)
display(hour_df)
display(price_stats_df)
display(price_quantiles)
display(session_summary_df)
display(session_df.head(20))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

if not event_type_df.empty:
    axes[0, 0].bar(event_type_df['event_type'], event_type_df['count'], color='#4C78A8')
    axes[0, 0].set_title('Event type distribution')
    axes[0, 0].set_xlabel('event_type')
    axes[0, 0].set_ylabel('count')

if not brand_df.empty:
    axes[0, 1].barh(brand_df['brand'][::-1], brand_df['count'][::-1], color='#F58518')
    axes[0, 1].set_title('Top 20 brands')
    axes[0, 1].set_xlabel('count')

if not month_df.empty:
    axes[1, 0].bar(month_df['month'], month_df['count'], color='#54A24B')
    axes[1, 0].set_title('Events by month')
    axes[1, 0].set_xlabel('month')
    axes[1, 0].tick_params(axis='x', rotation=45)

if not price_sample.empty:
    axes[1, 1].hist(price_sample.clip(upper=price_sample.quantile(0.99)), bins=60, color='#B279A2', edgecolor='white')
    axes[1, 1].set_title('Price distribution in sample (clipped at p99)')
    axes[1, 1].set_xlabel('price')
    axes[1, 1].set_ylabel('count')

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
if not hour_df.empty:
    ax.bar(hour_df['hour'], hour_df['count'], color='#72B7B2')
    ax.set_title('Events by hour of day')
    ax.set_xlabel('hour')
    ax.set_ylabel('count')
    ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## Ghi chú

- `category_id` là cột dư so với `RAW_SCHEMA`; notebook này không dùng nó cho EDA chính.
- Các thống kê session, duplicate, và quantile giá ở đây dựa trên sample ổn định từ mỗi file để giữ notebook chạy được trên raw source lớn.
- Nếu cần, bước tiếp theo là xuất phần summary này ra markdown report hoặc nâng cấp thành notebook chạy trên `data/raw/` sau khi materialize từ source.